# 📦 SARG LM GGUF Exporter & Google Drive Syncer

This notebook contains the automated pipeline to load your local fine-tuned LoRA weights (`Sarg_lm_lora.zip`), export them to a merged 4-bit GGUF model, and copy/sync them to your Google Drive safely without memory or sync corruption.

### Prerequisite:
Go to **Runtime > Change runtime type > T4 GPU** (or L4 / A100 if you have Colab Pro).

## 1. Install Unsloth and Dependencies
Run this cell to set up the environment.

> [!IMPORTANT]
> **CRITICAL STEP**: After running this cell, you **MUST** go to **Runtime > Restart Session** in the top menu. This resets Python's memory namespace so that the newly installed libraries don't trigger pickling errors during training.

In [ ]:
# Install Unsloth and compatible dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

## 2. Upload and Run the Automation Script

1. Zip your local **`Sarg_lm_lora`** folder on your computer into **`Sarg_lm_lora.zip`**.
2. Upload **`Sarg_lm_lora.zip`** to Colab's file browser (left sidebar).
3. Run the cell below. It will wait for the upload to complete, unzip the files, load the model, export it to GGUF, mount Google Drive, copy the GGUF model, and force-flush the sync to ensure it doesn't get corrupted.

In [ ]:
import os
import time
import zipfile
import shutil
import glob
from google.colab import drive
from unsloth import FastLanguageModel

zip_path = "Sarg_lm_lora.zip"

if not os.path.exists(zip_path):
    raise FileNotFoundError("Please upload 'Sarg_lm_lora.zip' to the left sidebar first!")

# 1. Wait for upload to complete
print("⏳ Monitoring upload progress...")
prev_size = -1
while True:
    current_size = os.path.getsize(zip_path)
    if current_size == prev_size and current_size > 0:
        print(f"\n✅ Upload complete! Size: {current_size / (1024**2):.2f} MB")
        break
    print(f"Uploading... Current size: {current_size / (1024**2):.2f} MB", end="\r")
    prev_size = current_size
    time.sleep(3)

# 2. Unzip files
print("📦 Unzipping files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("Sarg_lm_lora")

# 3. Load base model + LoRA weights in low-RAM mode
print("🤖 Loading fine-tuned model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Sarg_lm_lora/Sarg_lm_lora" if os.path.exists("Sarg_lm_lora/Sarg_lm_lora") else "Sarg_lm_lora", 
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = {"": 0}  # Forces GPU usage
)

# 4. Save GGUF model
print("🔄 Exporting to GGUF...")
model.save_pretrained_gguf("Sarg_lm_gguf", tokenizer, quantization_method = "q4_k_m")

# 5. Copy to Google Drive and Flush Sync
print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

gguf_files = glob.glob("**/*Q4_K_M.gguf", recursive=True)
if not gguf_files:
    raise FileNotFoundError("Could not find any GGUF files on the disk!")

# Find the file that isn't already inside the drive mount
source_path = [f for f in gguf_files if not f.startswith("drive")][0]
dest_path = "/content/drive/MyDrive/qwen2.5-7b-instruct.Q4_K_M.gguf"

print(f"⏳ Copying '{source_path}' to Google Drive...")
shutil.copy(source_path, dest_path)

print("💾 Flushing changes to Google Drive (this forces the upload to complete)...")
drive.flush_and_unmount()
print("🎉 SUCCESS! The file is now safely in Google Drive. You can download it locally now!")